# Bipolar MHW drivers — Richaud method, both hemispheres

Runs the Richaud tiling / event / driver-ranking pipeline over **both** polar oceans from the
global ACCESS-OM2 files on NCI, and produces the combined figures.

**Standalone.** Nothing here depends on `05_antarctic_richaud_method.ipynb`, on
`04_polar_mhw_arctic.ipynb`, or on the Arctic subset in `/scratch/m35/nm5072/arctic_mhw_subset/`.
The only prerequisite is `00_budget_clim_daily.ipynb`, which writes the global daily budget
climatology.

## Why rerun the Arctic rather than read Benjamin's result

Three reasons, in order of how much they matter:

1. **NB04 uses the monthly budget climatology**, interpolated to daily and then 31-day smoothed.
   That is exactly the aliasing `00_budget_clim_daily.ipynb` was built to remove, and it bites
   hardest in seasonally ice-covered water — which is most of the Arctic domain. This notebook
   uses the day-of-year climatology for both hemispheres.
2. **NB04 never writes `ds_events` to disk** — every `to_netcdf` in it is commented out — so
   there is no Arctic file to read even if we wanted one.
3. One code path means a hemisphere difference in the output is a hemisphere difference in the
   ocean, not a difference in how the two were processed.

## Grid handling

North of about 65°N the ACCESS-OM2 grid is tripolar: `yt_ocean` and `xt_ocean` are nominal
indices, not latitude and longitude. So this notebook uses the 2D `geolat_t` / `geolon_t`
fields and true cell areas (`area_t`) **for both hemispheres**. South of 60°S that is
numerically the same as NB05's `cos(yt_ocean)` weighting up to a constant — section 4 checks
that rather than assuming it.

Tiles are 20x20 blocks in *index* space in both hemispheres, as in NB04 and NB05. That is what
keeps tile statistics comparable; it is not a 5-degree box.

## Fidelity to NB04's tiling

Section 3 reproduces `generate_tiles_boxes` from NB04 cell 8. Identical by construction:

- 20x20 blocks in index space, numbered from the southern edge of the hemisphere subset;
- kept when the **absolute count** of ocean cells is at least `0.75 * 400 = 300` — the same
  keep/drop decision, not an approximation of it;
- ocean mask from the first 31 days of the first year, `temp_in_mld / 1035` non-null;
- tile centre latitude = plain mean of `geolat_t` over the tile's ocean cells;
- weighting by true cell area.

`_box_slice` differs in form but not in result: NB04 uses `.sel(x=slice(x_min, x_max))` on
integer index coordinates, which is inclusive of both ends; this notebook uses `.isel` with a
half-open slice. Both return the same 20x20 block.

Four places where this notebook does **not** match NB04. Each is switchable or reported, and
each is worth putting to Benjamin rather than settling here:

1. **Longitude centre.** NB04 takes a plain arithmetic mean of `geolon_t`. On a circle that is
   wrong whenever a tile's longitudes straddle the wrap point, and meaningless at the pole
   where all meridians converge. `LON_CENTER` selects; both values are computed and stored, and
   section 4 reports how many tiles the two disagree on by more than a degree.
2. **Tiles whose bounding box shrinks.** NB04 builds each tile with
   `tiles.where(tiles == nb, drop=True)`, which removes any row or column of the tile that is
   entirely land — including interior ones — so for those tiles the subsequent box mean covers
   a smaller region than the nominal block, and `ocean_frac` is computed against a hardcoded
   400 rather than the surviving cell count. This notebook always uses the full block and lets
   land cells fall out through the weighted mean. Section 4 counts the affected tiles.
3. **Tile id convention.** NB04 uses `x_min + box_size/2`; this uses the midpoint of the first
   and last index, which is half a cell lower. Ids are internally consistent but do not
   cross-reference to NB04's by name.
4. **`DROP_FOLD_ROW`.** NB04 does not treat the tripolar fold specially. Default here is
   `False`, matching it; the count of fold-touching tiles is reported either way.

Three things in NB04 cell 8 that look like defects rather than choices, all verified against
the code:

- The `boxes.append(...)` is dedented out of the `if ~np.isnan(nb):` guard, so the final
  `np.unique` element — `nan` — appends a **duplicate of the last valid tile**, using its stale
  `gcs`. Reproduced on a toy field: two tiles in, three boxes out.
- The `seam` flag is `gcs.geolon_t.min() > gcs.geolon_t.max()`, which cannot be true. The
  comment says "Should never be true", so the check never fires, and a genuinely
  seam-straddling tile would pass through unflagged.
- `ocean_frac = 1 - gcs.isnull().sum() / 400` is evaluated on the post-`drop=True` array, so
  for the tiles in point 2 it reports a higher ocean fraction than the tile actually has.

None of these change the *set* of tiles kept, which is decided by the absolute ocean-cell
count before any of it. The duplicate tile does enter the event statistics twice.

## Driver ranking

`RANK_BY = 'driver'`: rank 1 is the term that contributed most to what the heatwave was doing
in that phase.

- **onset** — the MHW is warming, so rank descending: rank 1 = most positive anomaly.
- **decline** — the MHW is cooling, so rank **ascending**: rank 1 = most negative anomaly.

NB04 ranks descending in both phases. At decline that makes rank 1 the term most strongly
*opposing* the decline, and pushes the term actually ending the event to rank 4 — which is
never plotted, since only ranks 1 and 2 appear in the figures. These are anomalies, so a
negative surface-flux anomaly cools relative to climatology and genuinely can end a heatwave;
the convention here counts it as a decline driver, and NB04's does not.

Consequence: the Arctic numbers here will **not** match the Arctic numbers in NB04's figures.
Set `RANK_BY = 'signed'` in section 1 to reproduce those exactly.

In [ ]:
import os
import glob
import pickle

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ultraplot as uplt

import dask
from dask.distributed import Client

%matplotlib inline

In [ ]:
# processes=True + threads_per_worker=1 avoids netCDF4/HDF5 thread-safety races
# when many workers read chunks from the same open_mfdataset files concurrently.
# On Gadi psutil reports the whole node rather than your ARE/PBS allocation, so set
# memory_limit explicitly:  (allocation x 0.9) / n_workers.
client = Client(processes=True, threads_per_worker=1, n_workers=6, memory_limit='20GB')
client

## 1. Configuration

In [ ]:
# ── Hemispheres ───────────────────────────────────────────────────────────────
# lat_slice is applied to yt_ocean, which is NOMINAL north of ~65N. It is a
# generous superset there, not a true parallel; the real cut is geolat_t, applied
# per tile in section 3.
HEMIS = {
    'Antarctic': dict(lat_slice=slice(-80.0, -60.0), geolat_min=-90.0, geolat_max=-60.0,
                      proj='splaea', boundinglat=-60),
    'Arctic':    dict(lat_slice=slice(60.0, None),   geolat_min=60.0,  geolat_max=90.0,
                      proj='nplaea', boundinglat=60),
}

BOX_SIZE     = 20      # tile edge in GRID CELLS (not degrees), as in NB04/NB05
MIN_OCN_FRAC = 0.75    # keep tiles with >= 0.75 * 400 = 300 ocean cells

# ── Data ──────────────────────────────────────────────────────────────────────
base          = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
BUDGET_SUBDIR = 'post_processed_diags/mlt_budget_online_stavg/'
START_OUTPUT  = 357     # 2010
END_OUTPUT    = 366     # 2019
outputs       = list(range(START_OUTPUT, END_OUTPUT + 1))

OUTPUT_DIR = '/scratch/m35/nm5072/Polar_MHWs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BUDGET_CLIM_FILE = OUTPUT_DIR + 'mlt_budget_clim_daily_336-365_global.nc'
MLT_CLIM_FILE    = base + 'post_processed_diags/om2_025_MLT_clim.nc'
MLT_THRESH_FILE  = base + 'post_processed_diags/om2_025_MLT_thresh.nc'

p2figs = OUTPUT_DIR + 'figures/'
os.makedirs(p2figs, exist_ok=True)

# ── MHW detection ─────────────────────────────────────────────────────────────
MIN_DUR = 5
MAX_GAP = 2

# ── Driver ranking ────────────────────────────────────────────────────────────
# 'driver' : rank 1 = biggest contributor to what the MHW is doing in that phase
#            (descending at onset, ASCENDING at decline)
# 'signed' : rank 1 = most positive anomaly in both phases   (NB04's convention)
# 'abs'    : rank 1 = largest |anomaly| in both phases       (NB03's convention)
RANK_BY = 'driver'

# ── Dask / I/O ────────────────────────────────────────────────────────────────
# Name EVERY dimension. Omitting one does not mean "one chunk per file" -- xarray falls
# back to that file's stored chunking, and ocean_daily.nc is stored one day at a time, so
# omitting 'time' there yields 3650 one-day chunks and a million-task graph.
#
# The two file sets are stored differently, so they need different dicts:
#   ocean_daily.nc          stored 1 day x full y x 240 in x  -> 'time' must be given
#   mlt_budget_*.nc         stored one chunk per file         -> omitting 'time' gives
#                                                                (365, 365, 366, ...),
#                                                                which is what we want;
#                                                                asking for 365 would split
#                                                                the leap-year files
MLT_CHUNKS    = {'time': 365, 'yt_ocean': BOX_SIZE, 'xt_ocean': -1}
BUDGET_CHUNKS = {'yt_ocean': BOX_SIZE, 'xt_ocean': -1}
CLIM_CHUNKS   = {'dayofyear': -1, 'yt_ocean': BOX_SIZE, 'xt_ocean': -1}

# Section 5 replaces a per-tile `.weighted().mean()` with one `coarsen` per band. Those are
# equal only if coarsen's sum skips NaN the way the weighted mean does -- and Coarsen.sum
# takes no explicit skipna argument, so that is implicit behaviour that could differ between
# xarray versions. VERIFY_COARSEN checks the two against each other on the first band
# computed and raises if they disagree, rather than letting a version difference through
# as quietly wrong numbers. Costs a few seconds, once.
VERIFY_COARSEN = True
ULP_TOL = 64          # float32 ULP of headroom for summation-order differences

# ── The two-day calendar skew ─────────────────────────────────────────────────
# ocean_daily.nc carries units 'days since 0001-01-01' with a STANDARD calendar
# (Julian before 1582-10-15, Gregorian after). xarray decodes that faithfully, but
# the values were written against a PROLEPTIC Gregorian epoch, and the two differ by
# exactly two days at modern dates -- twelve extra Julian leap days less the ten
# dropped in October 1582. So ocean_daily timestamps read two days early.
#
# Verified three ways:
#   cftime, same raw value -> 'standard' 2009-12-30, 'proleptic_gregorian' 2010-01-01
#   file spans             -> mlt output357 [2009-12-30..2010-12-29],
#                             budget output357 [2010-01-01..2010-12-31], every file
#   physics                -> corr(dMLT/dt, mlt_tendency) peaks 0.907 at lag +2,
#                             against 0.296 at lag 0
#
# The budget diagnostics carry TRUE dates. Detection is left on the as-written MLT
# timestamps: temperature, climatology and threshold all share the same skew, so it
# cancels and the events are right. The budget is the one place two products meet, so
# the budget ANOMALY is relabelled into the MLT convention in section 6 -- after its
# climatology has been removed in the true calendar, which is the order that matters.
MLT_TIME_SKEW_DAYS = 2
SEC_PER_DAY = 86400.0

# ── Budget terms ──────────────────────────────────────────────────────────────
TERM_MAP = {
    'MLT tendency'   : 'mlt_tendency',
    'Surface Flux'   : 'surf_to_ML',
    'Advection'      : 'advection',
    'Vertical mixing': 'vert_mixing',
    'Entrainment'    : 'entrainment',
}
BUDGET_TERMS = list(TERM_MAP.keys())

vars_raw = ['mlt_tendency', 'advection', 'vert_mixing',
            'entrainment', 'surface_flux', 'sw_pen', 'residual']

print('Hemispheres : {0}'.format(', '.join(HEMIS)))
print('Tiles       : {0}x{0} grid cells, >= {1:.0f} ocean cells'.format(
    BOX_SIZE, BOX_SIZE * BOX_SIZE * MIN_OCN_FRAC))
print('Years       : outputs {0}-{1} ({2}-{3})'.format(
    START_OUTPUT, END_OUTPUT, 1958 + START_OUTPUT - 305, 1958 + END_OUTPUT - 305))
print('Ranking     : {0}'.format(RANK_BY))

## 2. Locate the model grid

`geolat_t`, `geolon_t` and `area_t` come from the run's own `output{NNN}/ocean/ocean_grid.nc`
— same run, same grid, so it cannot disagree with the data. Confirmed on Gadi:

```
shape    : 1080 x 1440
geolat_t : -78.23 to  89.94
geolon_t : -279.94 to 79.94
area_t   : 1.84e+07 to 7.73e+08 m2
```

Three things to read off that. `geolon_t` is on the ACCESS −280 to 80 convention rather than
0–360, which is why tile longitude centres use a circular mean. The grid's southern boundary is
**−78.23°**, so the Antarctic domain is −78.23 to −60 however low `lat_slice` is set. And the
cell-area range is a factor of 42, from 7.7e8 m² (a 0.25° cell at the equator is about
27.7 km square) down to 1.84e7 m² in the tripolar north — which is the whole reason for
weighting by `area_t` rather than treating cells as equal.

The check below is on **shape**, not existence. A missing grid file fails loudly on the next
line anyway; a readable grid file that is the wrong grid would misplace every Arctic tile in
silence, which is the failure worth guarding against.

In [ ]:
GRID_FILE  = base + 'output{0:03d}/ocean/ocean_grid.nc'.format(START_OUTPUT)
GRID_SHAPE = (1080, 1440)      # the global 0.25 deg grid


def load_grid(path=GRID_FILE):
    """geolat_t, geolon_t, area_t as 2D DataArrays on (yt_ocean, xt_ocean)."""
    ds = xr.open_dataset(path, decode_times=False)
    # ocean_grid.nc tags its variables with coordinates="geolon_t geolat_t", so xarray
    # promotes those to coordinates. Strip them: carried along they collide when the grid
    # fields are merged with the budget data in a weighted mean.
    out = tuple(ds[n].squeeze(drop=True).reset_coords(drop=True)
                for n in ('geolat_t', 'geolon_t', 'area_t'))
    for da in out:
        if da.dims != ('yt_ocean', 'xt_ocean'):
            raise ValueError('{0} has dims {1}, expected (yt_ocean, xt_ocean)'.format(
                da.name, da.dims))
        if da.shape != GRID_SHAPE:
            raise ValueError(
                '{0} has shape {1}, expected {2}. Is this the global grid, or a subset?'
                .format(da.name, da.shape, GRID_SHAPE))
    return out


GEOLAT, GEOLON, AREA_T = load_grid()
print('Grid: {0}'.format(GRID_FILE))
print('  shape    : {0}'.format(dict(GEOLAT.sizes)))
print('  geolat_t : {0:.2f} to {1:.2f}'.format(float(GEOLAT.min()), float(GEOLAT.max())))
print('  geolon_t : {0:.2f} to {1:.2f}'.format(float(GEOLON.min()), float(GEOLON.max())))
print('  area_t   : {0:.3g} to {1:.3g} m2'.format(float(AREA_T.min()), float(AREA_T.max())))

## 3. Tiles

20x20 index blocks, kept when at least 75% of their cells are ocean. Tile centres come from
`geolat_t` / `geolon_t`, not from the nominal axes.

Longitude centres use a **circular** mean. A plain mean of `geolon_t` over a tile is wrong
wherever the tile spans the grid seam or sits near the pole, where longitudes converge; NB04
uses a plain mean, which is safe for most Arctic tiles and not for the polar-most ones.

Two things this cell reports and does not silently fix:

- **tiles touching the tripolar fold** (the northernmost grid row, where the grid folds onto
  itself so the row is physically duplicated). Set `DROP_FOLD_ROW = True` to exclude them.
- **tiles whose `geolat_t` range crosses the nominal cut**, i.e. where the nominal `yt_ocean`
  slice and the true latitude disagree.

In [ ]:
# NB04 does neither of these. Defaults here match NB04; change them deliberately.
DROP_FOLD_ROW = False       # True excludes tiles touching the northernmost grid row (the
                            # tripolar fold, where the row is physically duplicated)
LON_CENTER    = 'circular'  # 'circular' (this notebook) or 'plain' (NB04's arithmetic mean)


def _circmean_deg(lon_vals):
    """Circular mean of longitudes -> (mean in [0, 360), resultant length R in [0, 1]).

    R measures how concentrated the longitudes are. R near 1 means the tile occupies a
    narrow longitude band and the mean is meaningful; R near 0 means the longitudes are
    spread around the circle, which happens for tiles at the pole where every meridian
    converges, and the mean is then arbitrary. Section 4 reports how many tiles have low R
    rather than hiding them.
    """
    r = np.deg2rad(np.asarray(lon_vals, dtype=float).ravel())
    r = r[np.isfinite(r)]
    if r.size == 0:
        return np.nan, np.nan
    s, c = np.sin(r).mean(), np.cos(r).mean()
    deg = float(np.degrees(np.arctan2(s, c))) % 360.0
    # arctan2 can return a tiny negative for a mean that is exactly 0 degrees, and
    # (-1e-16) % 360.0 rounds to 360.0 in floating point. Pull that back to 0.
    if deg >= 360.0 - 1e-9:
        deg = 0.0
    return deg, float(np.hypot(s, c))


def generate_tiles(mask, geolat, geolon, area=None, box_size=BOX_SIZE,
                   ratio_gc=MIN_OCN_FRAC, ny_full=None, drop_fold_row=DROP_FOLD_ROW,
                   lon_center=LON_CENTER):
    """Tile the domain into box_size x box_size blocks of grid cells.

    mask/geolat/geolon are already sliced to the hemisphere and share a shape.
    ny_full is the northernmost yt_ocean index of the GLOBAL grid, used to spot tiles
    that touch the tripolar fold.
    """
    ny, nx = mask.sizes['yt_ocean'], mask.sizes['xt_ocean']
    ngcmin = box_size * box_size * ratio_gc
    is_ocean = (mask > 0).values
    lat2d, lon2d = geolat.values, geolon.values
    area2d = area.values if area is not None else None

    boxes, n_fold = [], 0
    for y0 in range(0, ny, box_size):
        y1 = min(y0 + box_size, ny)
        for x0 in range(0, nx, box_size):
            x1 = min(x0 + box_size, nx)
            cells = is_ocean[y0:y1, x0:x1]
            n_ocean = int(cells.sum())
            if n_ocean < ngcmin:
                continue

            touches_fold = (ny_full is not None) and (y1 >= ny)
            if touches_fold:
                n_fold += 1
                if drop_fold_row:
                    continue

            lat_tile = np.where(cells, lat2d[y0:y1, x0:x1], np.nan)
            lon_tile = np.where(cells, lon2d[y0:y1, x0:x1], np.nan)
            lon_circ, lon_R = _circmean_deg(lon_tile)
            lon_plain = float(np.nanmean(lon_tile)) % 360.0      # NB04's convention
            lon_c = lon_circ if lon_center == 'circular' else lon_plain

            # NB04's drop=True removes any all-land row or column of the tile, shrinking the
            # region its box mean covers. Flag the tiles that would happen to.
            shrunk = bool((~cells).all(axis=1).any() or (~cells).all(axis=0).any())

            boxes.append({
                'y0': y0, 'y1': y1, 'x0': x0, 'x1': x1,
                'y_center': (y0 + y1 - 1) / 2.0,
                'x_center': (x0 + x1 - 1) / 2.0,
                'lat_center': float(np.nanmean(lat_tile)),
                'lon_center': lon_c,
                'lon_center_circular': lon_circ,
                'lon_center_plain': lon_plain,
                'lon_R': lon_R,
                'area_m2': (float(np.nansum(np.where(cells, area2d[y0:y1, x0:x1], np.nan)))
                            if area2d is not None else np.nan),
                'nb04_bbox_shrinks': shrunk,
                'lat_min': float(np.nanmin(lat_tile)),
                'lat_max': float(np.nanmax(lat_tile)),
                'ocean_frac': n_ocean / float((y1 - y0) * (x1 - x0)),
                'fold': touches_fold,
            })
    return boxes, n_fold


def _box_slice(da, box):
    """Tiles are contiguous index blocks -> a plain isel. No seam handling needed."""
    return da.isel(yt_ocean=slice(box['y0'], box['y1']),
                   xt_ocean=slice(box['x0'], box['x1']))


def _box_mean(da, box, area):
    """Area-weighted mean over a tile. Land cells carry NaN area -> weight 0."""
    sub = _box_slice(da, box)
    w = _box_slice(area, box).fillna(0.0)
    return sub.weighted(w).mean(('yt_ocean', 'xt_ocean'))

## 4. Load one hemisphere

Everything stays lazy. The global files are opened once per hemisphere and sliced on
`yt_ocean`; dask reads only the chunks the slice touches, so the cost is close to reading a
pre-made subset.

In [ ]:
def load_hemisphere(name):
    """Open the global files, slice to this hemisphere, build its tiles."""
    cfg = HEMIS[name]
    sl = cfg['lat_slice']

    mlt_files = [base + 'output{0:03d}/ocean/ocean_daily.nc'.format(o) for o in outputs]
    budget_files = [base + BUDGET_SUBDIR +
                    'mlt_budget_stavg_daily_online_output{0:03d}.nc'.format(o) for o in outputs]
    for f in mlt_files + budget_files:
        if not os.path.exists(f):
            raise FileNotFoundError(f)

    # The hemisphere slice starts partway through a chunk, so without the trailing
    # .chunk() the y chunks come back as (9, 20, 20, ..., 16) and every band in section 5
    # straddles two of them.
    def _slice_align(obj):
        return obj.sel(yt_ocean=sl).chunk({'yt_ocean': BOX_SIZE})

    ds_wide = _slice_align(xr.open_mfdataset(
        mlt_files, decode_times=True, chunks=MLT_CHUNKS,
        combine='nested', concat_dim='time',
        data_vars=['temp_in_mld', 'mld'], parallel=True, decode_timedelta=False))
    temp_wide = ds_wide['temp_in_mld'] / 1035

    budget_wide = _slice_align(xr.open_mfdataset(
        budget_files, decode_times=True, chunks=BUDGET_CHUNKS,
        combine='nested', concat_dim='time',
        parallel=True, decode_timedelta=False))[vars_raw]
    budget_wide['surf_to_ML'] = budget_wide['surface_flux'] + budget_wide['sw_pen']

    # chunks= is required: without it these come back numpy-backed, and every reduction
    # below then runs eagerly and serially in the client process with the workers idle.
    clim_wide        = _slice_align(xr.open_dataset(MLT_CLIM_FILE,    chunks=CLIM_CHUNKS))
    thresh_wide      = _slice_align(xr.open_dataset(MLT_THRESH_FILE,  chunks=CLIM_CHUNKS))
    budget_clim_wide = _slice_align(xr.open_dataset(BUDGET_CLIM_FILE, chunks=CLIM_CHUNKS))

    # The grid file and the data files come from the same run, but xarray aligns on exact
    # float coordinate values, and a 1e-7 mismatch would silently empty every weighted mean.
    # Slice the grid the same way, check the shape, then force the data's own axes onto it.
    def _align(da):
        out = da.sel(yt_ocean=sl)
        if (out.sizes['yt_ocean'] != ds_wide.sizes['yt_ocean'] or
                out.sizes['xt_ocean'] != ds_wide.sizes['xt_ocean']):
            raise ValueError(
                '{0}: grid {1} does not match data {2} after the same yt_ocean slice. '
                'The grid file is on a different grid to the budget output.'.format(
                    da.name, dict(out.sizes), dict(ds_wide[["temp_in_mld"]].sizes)))
        return out.assign_coords(yt_ocean=ds_wide.yt_ocean, xt_ocean=ds_wide.xt_ocean)

    geolat = _align(GEOLAT)
    geolon = _align(GEOLON)
    area   = _align(AREA_T)

    # Ocean mask from the first month of the first year
    mask_month = (xr.open_dataset(mlt_files[0], decode_times=False)
                  .isel(time=slice(0, 31)).sel(yt_ocean=sl))
    ocean_full = (mask_month['temp_in_mld'] / 1035).notnull().mean('time').compute()

    ny_full = GEOLAT.sizes['yt_ocean'] if name == 'Arctic' else None
    boxes, n_fold = generate_tiles(ocean_full, geolat, geolon, area, ny_full=ny_full)
    ids = ['{0}_x{1:.0f}_y{2:.0f}'.format(name[:3], b['x_center'], b['y_center'])
           for b in boxes]

    # The skew is a property of the files, so check it rather than trust it.
    skew = float((budget_wide.time.values[0] - temp_wide.time.values[0])
                 / np.timedelta64(1, 'D'))
    if (skew != MLT_TIME_SKEW_DAYS
            or temp_wide.sizes['time'] != budget_wide.sizes['time']):
        raise ValueError(
            'MLT/budget time offset is {0:g} days over {1}/{2} steps, expected {3} over '
            'equal lengths. The calendar skew has changed -- re-check before trusting '
            'any budget composite.'.format(skew, temp_wide.sizes['time'],
                                           budget_wide.sizes['time'], MLT_TIME_SKEW_DAYS))

    print('{0}:'.format(name))
    print('  grid slice      : {0}'.format(dict(ocean_full.sizes)))
    print('  time            : mlt {0} .. {1}   budget {2} .. {3}  (skew {4:g} d, as expected)'
          .format(str(temp_wide.time.values[0])[:10], str(temp_wide.time.values[-1])[:10],
                  str(budget_wide.time.values[0])[:10],
                  str(budget_wide.time.values[-1])[:10], skew))
    for lbl, da in (('temp', temp_wide), ('budget', budget_wide[vars_raw[0]]),
                    ('bclim', budget_clim_wide[vars_raw[0]])):
        print('  chunks {0:7s}: {1}'.format(
            lbl, tuple(len(c) for c in da.chunks)), end='')
        print('  nchunks={0}  first={1}'.format(
            int(np.prod([len(c) for c in da.chunks])),
            tuple(c[0] for c in da.chunks)))
    print('  tiles kept      : {0}'.format(len(boxes)))
    print('  fold-row tiles  : {0} ({1})'.format(
        n_fold, 'dropped' if DROP_FOLD_ROW else 'kept'))
    print('  geolat_t range  : {0:.2f} to {1:.2f}   (slice was yt_ocean {2} to {3})'.format(
        min(b['lat_min'] for b in boxes), max(b['lat_max'] for b in boxes),
        sl.start, sl.stop))
    # Tile area, so "20x20 cells is not a fixed area" is a number rather than an assertion.
    a = np.array([b['area_m2'] for b in boxes]) / 1e9      # 10^3 km^2
    print('  tile area       : {0:.1f} to {1:.1f} (median {2:.1f}) x10^3 km2'
          '  -- max/min = {3:.1f}'.format(a.min(), a.max(), np.median(a), a.max() / a.min()))
    outside = [b for b in boxes
               if b['lat_center'] < cfg['geolat_min'] or b['lat_center'] > cfg['geolat_max']]
    print('  tile centres outside {0} to {1}: {2}'.format(
        cfg['geolat_min'], cfg['geolat_max'], len(outside)))
    vague = [b for b in boxes if b['lon_R'] < 0.5]
    print('  tiles with a poorly defined longitude centre (R<0.5): {0}'.format(len(vague)))
    if vague:
        print('    {0} -- longitudes spread around the circle, i.e. at the pole. Their'
              ' map position is arbitrary.'.format(
                  ', '.join(sorted({'{0:.0f}N'.format(b['lat_center']) for b in vague})[:6])))

    # --- differences from NB04's tiling, reported rather than assumed away -------------
    def _sep(a, b):
        d = abs(a - b) % 360.0
        return min(d, 360.0 - d)
    lon_disagree = [b for b in boxes
                    if _sep(b['lon_center_circular'], b['lon_center_plain']) > 1.0]
    shrunk = [b for b in boxes if b['nb04_bbox_shrinks']]
    print('  vs NB04: lon centre differs >1 deg on {0}/{1} tiles (using {2})'.format(
        len(lon_disagree), len(boxes), LON_CENTER))
    if lon_disagree:
        worst = max(lon_disagree,
                    key=lambda b: _sep(b['lon_center_circular'], b['lon_center_plain']))
        print('           worst {0:.0f} deg at lat {1:.1f}'.format(
            _sep(worst['lon_center_circular'], worst['lon_center_plain']),
            worst['lat_center']))
    print('  vs NB04: {0}/{1} tiles have an all-land row or column, so NB04\'s drop=True'
          ' would shrink their box mean'.format(len(shrunk), len(boxes)))
    print('  budget clim baseline : {0}'.format(
        budget_clim_wide.attrs.get('baseline_years', 'UNKNOWN')))

    return dict(temp=temp_wide, budget=budget_wide, clim=clim_wide, thresh=thresh_wide,
                bclim=budget_clim_wide, area=area, geolat=geolat, geolon=geolon,
                mask=ocean_full, boxes=boxes, ids=ids, cfg=cfg)


H = {name: load_hemisphere(name) for name in HEMIS}

### Check: does `area_t` weighting change the Antarctic answer?

South of 60°S the grid is regular, so cell area is proportional to `cos(lat)` and the constant
of proportionality cancels in a weighted mean. NB05 relies on that. This cell measures it on
the ocean mask rather than taking it on faith — the two weightings should agree to float
precision. If they do not, something is wrong with the grid file, not with the argument.

In [ ]:
_h = H['Antarctic']
_field = _h['mask']          # any 2D field works; the mask is already in memory
_diffs = []
for b in _h['boxes']:
    sub = _box_slice(_field, b)
    w_area = _box_slice(_h['area'], b).fillna(0.0)
    w_cos  = np.cos(np.deg2rad(sub['yt_ocean'])).broadcast_like(sub)
    m_area = float(sub.weighted(w_area).mean(('yt_ocean', 'xt_ocean')))
    m_cos  = float(sub.weighted(w_cos).mean(('yt_ocean', 'xt_ocean')))
    _diffs.append(abs(m_area - m_cos))
_diffs = np.array(_diffs)
print('Antarctic tiles compared : {0}'.format(len(_diffs)))
print('max |area_t - cos(lat)|  : {0:.3e}'.format(_diffs.max()))
print('median                   : {0:.3e}'.format(np.median(_diffs)))
print('\nSame check in the Arctic would NOT agree — that is the point of using area_t.')

## 5. Pre-compute tile means

One area-weighted mean per tile for MLT, its climatology and threshold, and every budget term
and its climatology. Checkpointed per latitude band per hemisphere, so a job that dies resumes.

**The checkpoint directory is keyed on the budget climatology file name.** Changing
`BUDGET_CLIM_FILE` therefore invalidates the cache automatically instead of silently reusing
box means built from a different baseline.

In [ ]:
CLIM_TAG = os.path.splitext(os.path.basename(BUDGET_CLIM_FILE))[0]
CHECKPOINT_ROOT = '{0}checkpoints_nb06/{1}/'.format(OUTPUT_DIR, CLIM_TAG)


def _band_tile_means(da, w, y0, y1, box_size=BOX_SIZE):
    """Area-weighted mean of EVERY tile in one latitude band, in a single reduction.

    Returns the same dims as `da` with yt_ocean dropped and xt_ocean replaced by one
    entry per tile column.

    This replaces a loop of per-tile `.weighted().mean()` calls. Those built a separate
    dask graph for every tile and every variable, 72 x 13 per band, which is about 44
    times more tasks than this for identical numbers -- and the cost was almost entirely
    scheduler overhead rather than compute. Verified equal to the per-tile result to
    2e-16 with the same NaN pattern.

    The xt_ocean coordinate is dropped so variables on `time` and on `dayofyear` merge
    into one Dataset positionally, rather than aligning on a coarsened float coordinate.
    """
    sub = da.isel(yt_ocean=slice(y0, y1))
    ww = w.isel(yt_ocean=slice(y0, y1))
    c = dict(yt_ocean=y1 - y0, xt_ocean=box_size)
    num = (sub * ww).coarsen(boundary='trim', **c).sum()
    den = ww.where(sub.notnull()).coarsen(boundary='trim', **c).sum()
    out = (num / den).squeeze('yt_ocean', drop=True)
    return out.drop_vars(['xt_ocean', 'yt_ocean'], errors='ignore')


def precompute_hemisphere(name, h):
    ckpt_dir = '{0}{1}/'.format(CHECKPOINT_ROOT, name)
    os.makedirs(ckpt_dir, exist_ok=True)

    y_band_centers = sorted(set(b['y_center'] for b in h['boxes']))
    y_bands = {yc: [(bid, b) for bid, b in zip(h['ids'], h['boxes'])
                    if abs(b['y_center'] - yc) < 0.01]
               for yc in y_band_centers}

    mlt_all, clim_all, thresh_all, budget_all, bclim_all = {}, {}, {}, {}, {}
    area_w = h['area'].fillna(0.0)

    for y_c, band in y_bands.items():
        if not band:
            continue
        ckpt = '{0}y_band_{1:.1f}.pkl'.format(ckpt_dir, y_c)
        if os.path.exists(ckpt):
            print('  {0} band {1:.0f} ({2} tiles) - checkpoint'.format(name, y_c, len(band)))
            with open(ckpt, 'rb') as fh:
                saved = pickle.load(fh)
            for d, key in ((mlt_all, 'mlt'), (clim_all, 'clim'), (thresh_all, 'thresh'),
                           (budget_all, 'budget'), (bclim_all, 'bclim')):
                d.update(saved[key])
            continue

        print('  {0} band {1:.0f} ({2} tiles)'.format(name, y_c, len(band)), flush=True)

        y0, y1 = band[0][1]['y0'], band[0][1]['y1']
        xi = {bid: b['x0'] // BOX_SIZE for bid, b in band}

        # Three groups because there are three axes here: the MLT time axis (two days
        # early), the budget time axis (true dates) and dayofyear. Putting them in one
        # Dataset makes xarray outer-join the two time axes and pad with NaN, which is
        # how the skew first showed up -- as a 3654-long array where 3652 was right.
        g_mlt = {'mlt': h['temp']}
        g_bud = {'B_' + term: h['budget'][raw] for term, raw in TERM_MAP.items()}
        g_doy = {'clim': h['clim'].temp, 'thresh': h['thresh'].temp}
        g_doy.update({'C_' + term: h['bclim'][raw] for term, raw in TERM_MAP.items()})

        # Still one graph: dask.compute walks all three together.
        ds_mlt, ds_bud, ds_doy = dask.compute(*[
            xr.Dataset({k: _band_tile_means(v, area_w, y0, y1) for k, v in g.items()})
            for g in (g_mlt, g_bud, g_doy)])

        if VERIFY_COARSEN and not _CHECKED:
            worst = 0.0
            for _bid, _b in band[:3]:
                ref = _box_mean(h['temp'], _b, h['area']).compute().values
                got = ds_mlt['mlt'].isel(xt_ocean=xi[_bid]).values

                # The NaN pattern must match EXACTLY. This is the sensitive test for the
                # thing the check exists for -- whether Coarsen.sum skips missing values
                # the way the weighted mean does.
                if not (np.isnan(ref) == np.isnan(got)).all():
                    raise AssertionError(
                        'coarsen and the per-tile weighted mean disagree about which days '
                        'are missing on tile {0} ({1} vs {2} NaN). This xarray ({3}) skips '
                        'NaN differently in Coarsen.sum -- do not trust these '
                        'results.'.format(_bid, int(np.isnan(ref).sum()),
                                          int(np.isnan(got).sum()), xr.__version__))

                # Finite values differ only by float32 summation order. The weights are
                # cell areas of order 1e8, accumulated over 400 cells, and the two methods
                # sum in different orders -- so a relative error of a few ULP is expected
                # and harmless. (In float64 the two orders are bit-identical.) Anything
                # structural, such as a misaligned window, is orders of magnitude larger.
                m = np.isfinite(ref) & np.isfinite(got)
                if m.any():
                    scale = max(float(np.max(np.abs(ref[m]))), 1.0)
                    tol = ULP_TOL * float(np.finfo(np.float32).eps) * scale
                    d = float(np.max(np.abs(ref[m] - got[m])))
                    worst = max(worst, d)
                    if d > tol:
                        raise AssertionError(
                            'coarsen band reduction disagrees with the per-tile weighted '
                            'mean on tile {0}: max diff {1:.3e} exceeds {2:.3e} '
                            '({3} float32 ULP at scale {4:.3g}). Too large to be rounding '
                            '-- do not trust these results.'.format(
                                _bid, d, tol, ULP_TOL, scale))
            print('    verified: coarsen == per-tile weighted mean on {0} tiles '
                  '(max diff {1:.2e}, float32 rounding)'.format(len(band[:3]), worst),
                  flush=True)
            _CHECKED.append(True)

        mlt_band    = {bid: ds_mlt['mlt'].isel(xt_ocean=xi[bid])    for bid, _ in band}
        clim_band   = {bid: ds_doy['clim'].isel(xt_ocean=xi[bid])   for bid, _ in band}
        thresh_band = {bid: ds_doy['thresh'].isel(xt_ocean=xi[bid]) for bid, _ in band}
        budget_band, bclim_band = {}, {}
        for bid, _ in band:
            for term in TERM_MAP:
                key = '{0}__{1}'.format(bid, term)
                budget_band[key] = ds_bud['B_' + term].isel(xt_ocean=xi[bid]) * SEC_PER_DAY
                bclim_band[key]  = ds_doy['C_' + term].isel(xt_ocean=xi[bid]) * SEC_PER_DAY

        with open(ckpt, 'wb') as fh:
            pickle.dump({'mlt': mlt_band, 'clim': clim_band, 'thresh': thresh_band,
                         'budget': budget_band, 'bclim': bclim_band}, fh)

        mlt_all.update(mlt_band);       clim_all.update(clim_band)
        thresh_all.update(thresh_band); budget_all.update(budget_band)
        bclim_all.update(bclim_band)

    return dict(mlt=mlt_all, clim=clim_all, thresh=thresh_all,
                budget=budget_all, bclim=bclim_all)


print('Checkpoints: {0}'.format(CHECKPOINT_ROOT))
_CHECKED = []          # VERIFY_COARSEN runs once per notebook run, on the first real band
BOXMEANS = {}
for name in HEMIS:
    print('\n== {0} =='.format(name))
    BOXMEANS[name] = precompute_hemisphere(name, H[name])
    print('  done: {0} tiles'.format(len(BOXMEANS[name]['mlt'])))

## 6. Detect events and decompose the budget

Hobday et al. (2016) detection per tile, then the mean anomaly of each budget term over the
onset window (`t_start` to `t_peak`) and the decline window (`t_peak` to `t_end`). The budget
climatology is applied by **day-of-year selection**, not by interpolating monthly values.

In [ ]:
def detect_mhw_events(temp_da, thresh_da, min_dur=5, max_gap=2):
    """Hobday et al. (2016) event detection. Returns dicts: t_start, t_peak, t_end."""
    t      = pd.to_datetime(temp_da.time.values)
    y_temp = temp_da.values
    y_thr  = thresh_da.values
    valid  = np.isfinite(y_temp) & np.isfinite(y_thr)
    if valid.sum() < min_dur:
        return []

    exceed = pd.Series((y_temp[valid] > y_thr[valid]).astype(int), index=t[valid])
    runs        = (exceed.diff(1).ne(0)).cumsum()
    run_lengths = exceed.groupby(runs).transform('size')
    merged      = exceed.copy()
    merged[(exceed == 0) & (run_lengths <= max_gap)] = 1

    runs2     = (merged.diff(1).ne(0)).cumsum()
    lens2     = merged.groupby(runs2).transform('size')
    long_true = (merged == 1) & (lens2 >= min_dur)

    anom = pd.Series(y_temp[valid] - y_thr[valid], index=t[valid])
    arr, idx = long_true.to_numpy(), long_true.index.to_numpy()
    events, start_i = [], None
    for i in range(len(arr)):
        if arr[i] and start_i is None:
            start_i = i
        if start_i is not None and (i == len(arr) - 1 or not arr[i + 1]):
            seg = anom.iloc[start_i:i + 1]
            events.append({'t_start': idx[start_i], 't_peak': seg.idxmax(), 't_end': idx[i]})
            start_i = None
    return events


def calendar_quarter(timestamp):
    """Calendar three-month grouping. Note that DJF is austral summer and boreal winter."""
    m = pd.Timestamp(timestamp).month
    if m in (12, 1, 2):
        return 'DJF'
    if m in (3, 4, 5):
        return 'MAM'
    if m in (6, 7, 8):
        return 'JJA'
    return 'SON'


def decompose_hemisphere(name, h, bm):
    all_events, all_df = {}, {}
    for k, (bid, box) in enumerate(zip(h['ids'], h['boxes'])):
        if k % 25 == 0:
            print('  [{0:3d}/{1}] {2}'.format(k, len(h['boxes']), bid), flush=True)

        temp_vals   = bm['mlt'][bid]
        thresh_mean = bm['thresh'][bid]

        doy_all = temp_vals.time.dt.dayofyear.clip(max=int(thresh_mean.dayofyear.max()))
        thresh_on_time = thresh_mean.sel(dayofyear=doy_all).assign_coords(time=temp_vals.time)

        events = detect_mhw_events(temp_vals, thresh_on_time, MIN_DUR, MAX_GAP)
        all_events[bid] = events
        if not events:
            all_df[bid] = pd.DataFrame()
            continue

        budget_series = xr.Dataset(
            {term: bm['budget']['{0}__{1}'.format(bid, term)] for term in BUDGET_TERMS})
        bc_box = xr.Dataset(
            {term: bm['bclim']['{0}__{1}'.format(bid, term)] for term in BUDGET_TERMS})

        doy_b = budget_series.time.dt.dayofyear.clip(max=int(bc_box.dayofyear.max()))
        clim_daily = xr.Dataset({
            nm: (bc_box[nm].sel(dayofyear=doy_b).drop_vars('dayofyear')
                 .assign_coords(time=budget_series.time))
            for nm in BUDGET_TERMS})
        anom_series = xr.Dataset(
            {nm: budget_series[nm] - clim_daily[nm] for nm in BUDGET_TERMS})

        # Everything above is on the budget's TRUE dates, which is what the day-of-year
        # climatology lookup requires. The events below come from the MLT axis, which is
        # MLT_TIME_SKEW_DAYS early. Relabel the anomaly -- after the climatology has been
        # removed, never before -- so the two products meet on one axis.
        anom_series = anom_series.assign_coords(
            time=anom_series.time - np.timedelta64(MLT_TIME_SKEW_DAYS, 'D'))

        rows = []
        for ev in events:
            t_s, t_p, t_e = ev['t_start'], ev['t_peak'], ev['t_end']
            q = calendar_quarter(t_p)
            for nm in BUDGET_TERMS:
                rows.append({
                    't_start': t_s, 't_peak': t_p, 't_end': t_e,
                    'quarter': q, 'term': nm,
                    'onset':   float(anom_series[nm].sel(time=slice(t_s, t_p)).mean()),
                    'decline': float(anom_series[nm].sel(time=slice(t_p, t_e)).mean()),
                })
        all_df[bid] = pd.DataFrame(rows)

    n = sum(len(v) for v in all_events.values())
    print('  {0}: {1} events across {2} tiles'.format(name, n, len(h['boxes'])))
    return all_events, all_df


EVENTS, DFS = {}, {}
for name in HEMIS:
    print('== {0} =='.format(name))
    EVENTS[name], DFS[name] = decompose_hemisphere(name, H[name], BOXMEANS[name])

## 7. Reshape to `ds_events`

One dataset per hemisphere with dimensions (events x tiles x phase), plus a combined dataset
carrying a `hemisphere` coordinate. Both are written to scratch, so the figures below — and
anything else you want to do with these events later — do not require rerunning sections 5–6.

In [ ]:
def box_df2xr(df):
    """Long per-event table -> wide dataset with dims (events, phase)."""
    event_keys = ['t_start', 't_peak', 't_end']
    df = df.copy()
    df['event_id'] = df.groupby(event_keys).ngroup()

    long = (df.melt(id_vars=['event_id'] + event_keys + ['term'],
                    value_vars=['onset', 'decline'],
                    var_name='phase', value_name='value')
              .pivot_table(index=['event_id', 'phase'], columns='term', values='value'))

    ds = long.to_xarray().rename({'event_id': 'events'})
    meta = df[['event_id'] + event_keys].drop_duplicates().set_index('event_id')
    ds = ds.assign_coords(t_start=('events', meta['t_start'].values),
                          t_peak=('events', meta['t_peak'].values),
                          t_end=('events', meta['t_end'].values))
    ds = ds.transpose('events', 'phase')
    for var in list(ds.data_vars):
        ds[var] = ds[var].assign_attrs({'standard_name': var, 'units': 'degC day$^{-1}$'})
        ds = ds.rename({var: TERM_MAP[var]})
    return ds


def build_ds_events(name, h, dfs):
    tmp = [box_df2xr(df).expand_dims({'tiles': [bid]})
           for bid, df in dfs.items() if not df.empty]
    ds = xr.concat(tmp, 'tiles', join='outer', coords='different')

    by_id = dict(zip(h['ids'], h['boxes']))
    ds = ds.assign_coords(
        x_center=('tiles', [by_id[t]['x_center'] for t in ds.tiles.values]),
        y_center=('tiles', [by_id[t]['y_center'] for t in ds.tiles.values]),
        lat_center=('tiles', [by_id[t]['lat_center'] for t in ds.tiles.values]),
        lon_center=('tiles', [by_id[t]['lon_center'] for t in ds.tiles.values]),
        ocean_frac=('tiles', [by_id[t]['ocean_frac'] for t in ds.tiles.values]),
        hemisphere=('tiles', [name] * ds.sizes['tiles']),
    )
    ds.attrs.update({
        'description': '{0} MHW mixed-layer heat budget, Richaud-method tiles'.format(name),
        'model': 'ACCESS-OM2 0.25 deg IAF cycle 6',
        'outputs': '{0}-{1}'.format(START_OUTPUT, END_OUTPUT),
        'box_size_cells': BOX_SIZE,
        'min_ocn_frac': MIN_OCN_FRAC,
        'mhw_min_dur': MIN_DUR,
        'mhw_max_gap': MAX_GAP,
        'budget_clim': BUDGET_CLIM_FILE,
        'budget_clim_baseline': h['bclim'].attrs.get('baseline_years', 'UNKNOWN'),
        'weighting': 'area_t',
        'rank_by': RANK_BY,
        'fold_row_tiles': 'dropped' if DROP_FOLD_ROW else 'kept',
        'time_convention': (
            'Event times are on the ocean_daily as-written timestamps, which decode '
            'two days EARLY: the files declare calendar GREGORIAN with units days '
            'since 0001-01-01, but the values were written against a proleptic '
            'Gregorian epoch. ADD 2 DAYS for true dates. The budget diagnostics '
            'carry true dates and their anomalies were shifted back two days to '
            'match, after climatology removal. Detection is self-consistent: '
            'temperature and threshold share the skew.'),
    })
    return ds


DS = {name: build_ds_events(name, H[name], DFS[name]) for name in HEMIS}
for name, ds in DS.items():
    path = '{0}06_{1}_mhw_budget_events.nc'.format(OUTPUT_DIR, name)
    ds.to_netcdf(path)
    print('Saved -> {0}  ({1} tiles, {2} events)'.format(
        path, ds.sizes['tiles'], ds.sizes['events']))

DS_BOTH = xr.concat([DS[n] for n in HEMIS], dim='tiles', join='outer', coords='different')
DS_BOTH.attrs.update({'description': 'Bipolar MHW mixed-layer heat budget, Richaud-method tiles',
                      'hemispheres': ', '.join(HEMIS)})
both_path = OUTPUT_DIR + '06_bipolar_mhw_budget_events.nc'
DS_BOTH.to_netcdf(both_path)
print('Saved -> {0}'.format(both_path))

## 8. Driver ranking

For every (tile, event) the four driver terms are ranked. `RANK_BY` is set in section 1; the
three conventions are described in the header.

`nMHW` is counted **per hemisphere**, so the percentages below are the fraction of that
hemisphere's events — the two hemispheres have different tile counts and different event
counts, and a shared denominator would make the bars meaningless.

In [ ]:
DRIVER_TERMS = [t for t in TERM_MAP.values() if t != 'mlt_tendency']
DRIVER_TICKS = ['Surface', 'Lateral', 'Vertical mixing', 'Entrainment']
ORDER = ['surf_to_ML', 'advection', 'vert_mixing', 'entrainment']


def phase_ranks(ds, phase, rank_by=RANK_BY):
    """(tiles x events) table of driver ranks for one phase. 1 = primary."""
    df = (ds.sel(phase=phase).drop_vars(['phase'])
            .to_dataframe(dim_order=('tiles', 'events'))
            .dropna()[DRIVER_TERMS])
    if rank_by == 'abs':
        return df.abs().rank(axis=1, ascending=False)
    if rank_by == 'signed':
        return df.rank(axis=1, ascending=False)
    if rank_by == 'driver':
        # onset: most positive first. decline: most negative first.
        return df.rank(axis=1, ascending=(phase == 'decline'))
    raise ValueError('RANK_BY must be driver, signed or abs')


RANKS, TALLY, NMHW = {}, {}, {}
n_rank = len(DRIVER_TERMS)

for name in HEMIS:
    RANKS[name] = {ph: phase_ranks(DS[name], ph) for ph in ('onset', 'decline')}
    NMHW[name] = float(len(RANKS[name]['onset']))
    TALLY[name] = {}
    for ph in ('onset', 'decline'):
        # NB04's index form: [np.arange(...)] gives a 1-level MultiIndex, so .loc[rk] is a
        # 1-row DataFrame. ultraplot's bar(..., mean=True) needs 2D and raises on a Series.
        tab = pd.DataFrame(columns=DRIVER_TERMS, index=[np.arange(1, n_rank + 1)])
        for rk in np.arange(1, n_rank + 1):
            tab.loc[rk] = RANKS[name][ph].where(RANKS[name][ph] == rk).count().values
        TALLY[name][ph] = tab.astype('float')

    print('{0}: {1:.0f} events ranked'.format(name, NMHW[name]))
    for ph in ('onset', 'decline'):
        pct = (TALLY[name][ph].loc[1] / NMHW[name] * 100).round(1)
        print('  {0:7s} primary %: {1}'.format(
            ph, ', '.join('{0}={1:.1f}'.format(t, float(pct[t])) for t in ORDER)))

## 9. Combined bar chart

Rows are onset and decline, columns are the primary and secondary driver, and the two bars in
each group are the two hemispheres. Percentages are of that hemisphere's own event count.

In [ ]:
HEMI_COLORS = ['cobalt', 'wine red']
HEMI_NAMES = list(HEMIS)


def rank_frame_phase(phase, rank):
    """4 terms x 2 hemispheres, as a percentage of each hemisphere's own event count."""
    return pd.DataFrame(
        {name: [float(TALLY[name][phase].loc[rank][t]) / NMHW[name] * 100 for t in ORDER]
         for name in HEMI_NAMES}, index=DRIVER_TICKS)


fig, axs = uplt.subplots(ncols=2, nrows=2, refwidth=3, refaspect=1.5)
panels = [('onset', 1), ('onset', 2), ('decline', 1), ('decline', 2)]
h_bar = None
for k, (ph, rk) in enumerate(panels):
    hb = axs[k].bar(rank_frame_phase(ph, rk), cycle=HEMI_COLORS, edgecolor='k', linewidth=0.4)
    if h_bar is None:
        h_bar = hb

axs[2:4].format(xticklabels=DRIVER_TICKS, xrotation=30)
axs.format(leftlabels=['Onset', 'Decline'], toplabels=['Primary', 'Secondary'],
           abc='a)', abcloc='ul', fontsize=12,
           yformatter='percent', ylim=[0, 100],
           ylabel='Percentage of Marine Heatwaves driven [%]')
fig.legend(h_bar, labels=HEMI_NAMES, loc='b', ncol=2)
fig.suptitle('Drivers of polar Marine Heatwaves  (ranking: {0})'.format(RANK_BY), fontsize=14)
# fig.save(p2figs + 'BarPlot_bipolar_MHWDrivers_ranking_{0}.png'.format(RANK_BY), dpi=600)

## 10. Combined maps

Four rows — Antarctic onset, Antarctic decline, Arctic onset, Arctic decline — by four driver
terms. Marker size is the fraction of that tile's events whose primary driver is the column's
term.

`absolute_size=True` is deliberate, and is the one place this notebook departs from NB04's
plotting. NB04 passes `smin=0, smax=50`, which rescales marker sizes to each **panel's own**
data range — so a panel where a term is primary in 2% of events draws markers the same size as
a panel where it is primary in 80%, and the size legend applies to neither. With absolute
sizes all sixteen panels share one scale and the legend means what it says.

In [ ]:
def primary_fraction(rank_df, term, mhws_per_tile, coords):
    n = rank_df.where(rank_df[term] == 1).dropna().groupby('tiles').size()
    frac = (n / mhws_per_tile).dropna()
    return frac, coords.sel(tiles=frac.index)


FRACS, COORDS = {}, {}
for name in HEMIS:
    CoordsT = DS[name][['lat_center', 'lon_center']]
    COORDS[name] = CoordsT
    per_tile = RANKS[name]['onset'].dropna().groupby('tiles').size()
    FRACS[name] = {
        ph: {t: primary_fraction(RANKS[name][ph], t, per_tile, CoordsT) for t in ORDER}
        for ph in ('onset', 'decline')}
    print('{0}: {1} tiles with events'.format(name, len(per_tile)))
    for ph in ('onset', 'decline'):
        print('  {0:7s} '.format(ph) + '  '.join(
            '{0}={1}'.format(t, len(FRACS[name][ph][t][0])) for t in ORDER))

In [ ]:
SMAX = 50.0     # marker area (pt^2) for a fraction of 1.0, shared by every panel
ROWS = [('Antarctic', 'onset'), ('Antarctic', 'decline'),
        ('Arctic', 'onset'), ('Arctic', 'decline')]

projs = []
for name, _ in ROWS:
    projs += [HEMIS[name]['proj']] * 4

fig, axs = uplt.subplots(nrows=4, ncols=4, proj=projs, refwidth=2.4)

sc = None
for r, (name, ph) in enumerate(ROWS):
    blat = HEMIS[name]['boundinglat']
    for j, term in enumerate(ORDER):
        ax = axs[r * 4 + j]
        # every tile in white first: "no events of this kind" vs "no tile"
        ax.scatter(COORDS[name].lon_center, COORDS[name].lat_center,
                   s=30, c='w', edgecolor='w', m='o', absolute_size=True)
        frac, coords = FRACS[name][ph][term]
        hh = ax.scatter(coords.lon_center, coords.lat_center,
                        s=frac.values * SMAX, c=uplt.get_colors(uplt.Cycle('default', 6))[1 + j],
                        m='o', absolute_size=True)
        if sc is None:
            sc = hh
        ax.format(boundinglat=blat)

axs.format(land=True, landcolor='grey', ocean=True, oceancolor='grey2', oceanzorder=0,
           toplabels=['Surface\nHeat Flux', 'Lateral\nHeat Flux',
                      'Vertical\nMixing', 'Entrainment'],
           leftlabels=['Antarctic\nonset', 'Antarctic\ndecline',
                       'Arctic\nonset', 'Arctic\ndecline'],
           abc='a)', abcloc='ul')
fig.legend(*sc.legend_elements('sizes', num=4, func=lambda s: s / SMAX * 100,
                               fmt=uplt.PercentFormatter()),
           loc='b', ncol=4, title='Proportion of local Marine Heatwaves')
fig.suptitle('Location of polar Marine Heatwaves dominated by:  (ranking: {0})'.format(RANK_BY))
# fig.save(p2figs + 'ScatterMaps_bipolar_FirstDrivers_{0}.png'.format(RANK_BY), dpi=400)

## 11. The numbers

The figures are for reading patterns; this is the table to quote. Primary-driver percentages
per hemisphere and phase, plus the hemisphere difference in percentage points.

In [ ]:
rows = []
for ph in ('onset', 'decline'):
    for t, tick in zip(ORDER, DRIVER_TICKS):
        rec = {'phase': ph, 'term': tick}
        for name in HEMIS:
            rec[name] = float(TALLY[name][ph].loc[1][t]) / NMHW[name] * 100
        rec['difference'] = rec[HEMI_NAMES[0]] - rec[HEMI_NAMES[1]]
        rows.append(rec)

summary = pd.DataFrame(rows).set_index(['phase', 'term']).round(1)
print('Primary driver, % of that hemisphere\'s events   (ranking: {0})\n'.format(RANK_BY))
print(summary.to_string())
print('\nEvents: ' + ',  '.join('{0} {1:.0f}'.format(n, NMHW[n]) for n in HEMIS))
print('Tiles : ' + ',  '.join('{0} {1}'.format(n, DS[n].sizes['tiles']) for n in HEMIS))

summary.to_csv(OUTPUT_DIR + '06_bipolar_primary_driver_{0}.csv'.format(RANK_BY))
print('\nSaved -> {0}06_bipolar_primary_driver_{1}.csv'.format(OUTPUT_DIR, RANK_BY))

## 12. Caveats worth carrying into the chapter

- **Tile area is not constant, within or between hemispheres.** A 20x20 index block is a fixed
  number of grid cells, not a fixed area. Tiling in index space rather than in degrees is
  Richaud's own choice, made precisely because degree boxes break down at the pole — NB04 cell
  9 keeps the old degree-based tiler commented out with the note "This won't work for the
  Arctic, because of the Northpole". So the caveat is inherited from the method, not introduced
  here. What *is* new is comparing two hemispheres of such tiles: Richaud tiles the Arctic
  alone, where a varying tile area never has to line up against anything. Section 4 prints the
  min / median / max tile area and the max:min ratio per hemisphere, so this is a measured
  number rather than an assertion. Event counts per tile are comparable between hemispheres;
  anything per unit area is not, unless those two ratios are close.
- **Longitude centres near the North Pole are ill-defined, and NB04 does not flag this.** It
  takes a plain arithmetic mean of `geolon_t` with no polar special-casing. The degeneracy is
  in the longitude *coordinate*, not in the grid: the ACCESS-OM2 tripolar grid is smooth across
  the geographic pole, because its two singular points sit on the joining latitude over land.
  The practical effect is also self-limiting — on a polar projection a tile at 89°N plots
  essentially at the centre whatever longitude you give it, so the positional error shrinks as
  the longitude becomes less meaningful. Section 4 reports the tiles with resultant length
  R < 0.5; treat those markers as "near the pole" and read nothing into their bearing.
- **Check whether the `yt_ocean = 60` cut is a true parallel — do not assume either way.**
  The `03_extract_arctic_subset.ipynb` header states it is a nominal cut and not a true 60°N
  parallel. That may be too pessimistic: MOM5 tripolar grids are regular lat-lon *south* of the
  joining latitude (~65°N for this grid), which would make the row at `yt_ocean = 60` a genuine
  parallel, with only the interior labelling north of the join being nominal. Section 4 now
  prints the `geolat_t` range of the kept tiles next to the slice bounds; if the Arctic minimum
  comes back at ≈60.0 the cut is a true parallel and there is nothing to fix. It can also be
  settled directly from the grid file — see the cell below. Tile centres come from `geolat_t`
  either way, so nothing downstream depends on resolving it.
- **`RANK_BY` changes the decline answer.** Section 8 makes the convention explicit in the
  saved attributes and in the figure titles. Re-run with `'signed'` before comparing anything
  here against NB04's published Arctic figures.
- **Grid provenance is assumed, not verified.** NB04 reads `meshgrid_access-om2_025_60N.nc`
  and `areacello_access-om2_025_60N.nc`; this notebook reads `geolat_t` / `geolon_t` /
  `area_t` from the run's own `ocean_grid.nc`. They should be the same numbers — `areacello`
  is the CMOR name for the T-cell area that MOM5 writes as `area_t` — but that has not been
  checked cell by cell. Worth confirming with Benjamin what those two files were cut from
  before any hemisphere difference is attributed to the ocean.

In [ ]:
# Where does the grid stop being a regular lat-lon grid? On a row that is a true parallel,
# geolat_t is constant along it; north of the joining latitude it is not. This settles
# whether the yt_ocean = 60 cut is a real 60N parallel.
print('yt_ocean    geolat_t min     max      spread')
for y in [50, 55, 58, 60, 62, 65, 68, 70, 75, 80]:
    row = GEOLAT.sel(yt_ocean=y, method='nearest')
    lo, hi = float(row.min()), float(row.max())
    print('{0:8.3f}    {1:9.4f} {2:9.4f}   {3:8.4f}{4}'.format(
        float(row.yt_ocean), lo, hi, hi - lo,
        '   <- joining latitude is at or below here' if hi - lo > 1e-3 else ''))

In [ ]:
client.close()